<a href="https://colab.research.google.com/github/Prajwala15/Prajwala15/blob/main/Reading_Text_in_Images.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%cd /content
!git clone https://github.com/Prajwala15/Prajwala15.git textvqa
%cd textvqa
!ls -la

/content
Cloning into 'textvqa'...
remote: Enumerating objects: 25, done.
remote: Counting objects: 100% (25/25), done.
remote: Compressing objects: 100% (22/22), done.
remote: Total 25 (delta 0), reused 25 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (25/25), 14.59 KiB | 2.92 MiB/s, done.
/content/textvqa
total 44
drwxr-xr-x 8 root root 4096 Jun 21 12:01 .
drwxr-xr-x 1 root root 4096 Jun 21 12:01 ..
drwxr-xr-x 2 root root 4096 Jun 21 12:01 configs
drwxr-xr-x 2 root root 4096 Jun 21 12:01 data
drwxr-xr-x 8 root root 4096 Jun 21 12:01 .git
-rw-r--r-- 1 root root   69 Jun 21 12:01 .gitignore
drwxr-xr-x 2 root root 4096 Jun 21 12:01 outputs
-rw-r--r-- 1 root root 3169 Jun 21 12:01 README.md
-rw-r--r-- 1 root root  303 Jun 21 12:01 requirements.txt
drwxr-xr-x 2 root root 4096 Jun 21 12:01 scripts
drwxr-xr-x 3 root root 4096 Jun 21 12:01 src


In [2]:
!pip install -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 45.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 929.8/929.8 kB 28.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.2/978.2 kB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 299.6/299.6 kB 16.9 MB/s eta 0:00:00


In [3]:
!ls -la

total 44
drwxr-xr-x 8 root root 4096 Jun 21 12:01 .
drwxr-xr-x 1 root root 4096 Jun 21 12:01 ..
drwxr-xr-x 2 root root 4096 Jun 21 12:01 configs
drwxr-xr-x 2 root root 4096 Jun 21 12:01 data
drwxr-xr-x 8 root root 4096 Jun 21 12:01 .git
-rw-r--r-- 1 root root   69 Jun 21 12:01 .gitignore
drwxr-xr-x 2 root root 4096 Jun 21 12:01 outputs
-rw-r--r-- 1 root root 3169 Jun 21 12:01 README.md
-rw-r--r-- 1 root root  303 Jun 21 12:01 requirements.txt
drwxr-xr-x 2 root root 4096 Jun 21 12:01 scripts
drwxr-xr-x 3 root root 4096 Jun 21 12:01 src


In [4]:
!ls scripts

00_download.md	01_run_ocr.py  02_eval.py  03_analysis.py


In [5]:
!find data -type f | head -20

data/.gitkeep


In [6]:
!ls scripts

00_download.md	01_run_ocr.py  02_eval.py  03_analysis.py


In [7]:
!cat scripts/00_download.md

# Getting the TextVQA data

The sandbox has no internet for these hosts, so download on your own machine.

## Annotations (small)
From https://textvqa.org/dataset/ :
- `TextVQA_0.5.1_train.json`
- `TextVQA_0.5.1_val.json`
- `TextVQA_0.5.1_test.json`

Put them in `data/`.

## Images (~25 GB)
TextVQA images come from OpenImages. The dataset page provides:
- `train_val_images.zip`  -> unzip to `data/train_images/`
- `test_images.zip`       -> unzip to `data/test_images/`

(Val images are inside the train_val set; the loader resolves paths by `image_id`.)

## Alternative: Hugging Face
```python
from datasets import load_dataset
ds = load_dataset("textvqa")          # has images + questions + answers
```
If you use HF instead of the raw json, swap `src/dataset.py`'s loader for one that
iterates the HF split — the rest of the pipeline only needs each sample to expose
`question_id, question, image (PIL), answers (list[str])`.

## Sanity check
```bash
python -c "from src.dataset import TextVQA

In [8]:
!pip install -r requirements.txt

In [9]:
!cat configs/default.yaml

# Default configuration. Override on the CLI where supported.

paths:
  data_root: data
  # TextVQA annotation files (download separately, see scripts/00_download.md)
  ann:
    train: data/TextVQA_0.5.1_train.json
    val:   data/TextVQA_0.5.1_val.json
    test:  data/TextVQA_0.5.1_test.json
  # image folders (train_images, test_images from open-images subset)
  images:
    train: data/train_images
    val:   data/train_images   # val images live in the train_images folder for TextVQA 0.5.1
    test:  data/test_images
  ocr_cache: outputs/ocr_cache   # one json per split
  preds:     outputs/preds
  scores:    outputs/scores

ocr:
  engine: easyocr        # easyocr | tesseract
  langs: [en]
  max_tokens: 50         # cap OCR tokens fed to the reasoner
  min_confidence: 0.30

vlm:
  backend: blip2         # blip2 | api
  blip2_model: Salesforce/blip2-flan-t5-xl
  api_model: claude-sonnet-4-6     # only used when backend: api
  max_new_tokens: 16

ocr_first:
  # backend that turns (ques

In [10]:
!python --version

Python 3.12.13


In [11]:
!ls -lh data

total 0


In [23]:
import os
os.chdir("/content/textvqa")
!cat src/config.py

import os
import yaml

_DEFAULT = os.path.join(os.path.dirname(os.path.dirname(__file__)), "configs", "default.yaml")


def load_config(path: str = _DEFAULT) -> dict:
    with open(path) as f:
        return yaml.safe_load(f)


def pick_device(pref: str = "auto") -> str:
    if pref != "auto":
        return pref
    try:
        import torch
        return "cuda" if torch.cuda.is_available() else "cpu"
    except Exception:
        return "cpu"


In [17]:
!cat configs/default.yaml

# Default configuration. Override on the CLI where supported.

paths:
  data_root: data
  # TextVQA annotation files (download separately, see scripts/00_download.md)
  ann:
    train: data/TextVQA_0.5.1_train.json
    val:   data/TextVQA_0.5.1_val.json
    test:  data/TextVQA_0.5.1_test.json
  # image folders (train_images, test_images from open-images subset)
  images:
    train: data/train_images
    val:   data/train_images   # val images live in the train_images folder for TextVQA 0.5.1
    test:  data/test_images
  ocr_cache: outputs/ocr_cache   # one json per split
  preds:     outputs/preds
  scores:    outputs/scores

ocr:
  engine: easyocr        # easyocr | tesseract
  langs: [en]
  max_tokens: 50         # cap OCR tokens fed to the reasoner
  min_confidence: 0.30

vlm:
  backend: blip2         # blip2 | api
  blip2_model: Salesforce/blip2-flan-t5-xl
  api_model: claude-sonnet-4-6     # only used when backend: api
  max_new_tokens: 16

ocr_first:
  # backend that turns (ques

In [24]:
!sed -n '1,200p' src/dataset.py

"""TextVQA dataset loader (TextVQA 0.5.1 raw-json format).

Each item exposes:
    question_id : int
    image_id    : str
    question    : str
    image_path  : str
    answers     : list[str]   (10 human answers; empty for test split)
"""
import json
import os
from functools import lru_cache

from PIL import Image

from .config import load_config


def _resolve_image_path(image_dir: str, image_id: str) -> str:
    # TextVQA image_id maps to <image_id>.jpg in the open-images folder.
    for ext in (".jpg", ".jpeg", ".png"):
        p = os.path.join(image_dir, image_id + ext)
        if os.path.exists(p):
            return p
    # fall back to the bare id (caller handles missing files)
    return os.path.join(image_dir, image_id + ".jpg")


class TextVQADataset:
    def __init__(self, split: str = "val", cfg: dict | None = None, limit: int = 0):
        self.cfg = cfg or load_config()
        self.split = split
        ann_path = self.cfg["paths"]["ann"][split]
        self.image_dir

In [26]:
%cd /content/textvqa/data

!wget https://dl.fbaipublicfiles.com/textvqa/data/TextVQA_0.5.1_train.json
!wget https://dl.fbaipublicfiles.com/textvqa/data/TextVQA_0.5.1_val.json
!wget https://dl.fbaipublicfiles.com/textvqa/data/TextVQA_0.5.1_test.json

!ls -lh

/content/textvqa/data
--2026-06-21 12:58:33--  https://dl.fbaipublicfiles.com/textvqa/data/TextVQA_0.5.1_train.json
Resolving dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)... 3.167.112.66, 3.167.112.53, 3.167.112.129, ...
Connecting to dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)|3.167.112.66|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 21634937 (21M) [text/plain]
Saving to: ‘TextVQA_0.5.1_train.json’

TextVQA_0.5.1_train 100%[===================>]  20.63M  --.-KB/s    in 0.05s   

2026-06-21 12:58:33 (376 MB/s) - ‘TextVQA_0.5.1_train.json’ saved [21634937/21634937]

--2026-06-21 12:58:33--  https://dl.fbaipublicfiles.com/textvqa/data/TextVQA_0.5.1_val.json
Resolving dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)... 3.167.112.66, 3.167.112.53, 3.167.112.129, ...
Connecting to dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)|3.167.112.66|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 3116162 (3.0M) [text/plain]
Saving to: ‘Tex

In [27]:
!ls -lh data

ls: cannot access 'data': No such file or directory


In [28]:
!pwd
!ls -la

/content/textvqa/data
total 26888
drwxr-xr-x 2 root root     4096 Jun 21 12:58 .
drwxr-xr-x 8 root root     4096 Jun 21 12:01 ..
-rw-r--r-- 1 root root        0 Jun 21 12:01 .gitkeep
-rw-r--r-- 1 root root  2770520 Mar 16  2020 TextVQA_0.5.1_test.json
-rw-r--r-- 1 root root 21634937 Mar 16  2020 TextVQA_0.5.1_train.json
-rw-r--r-- 1 root root  3116162 Mar 16  2020 TextVQA_0.5.1_val.json


In [29]:
%cd /content/textvqa
!pwd
!ls -la

/content/textvqa
/content/textvqa
total 44
drwxr-xr-x 8 root root 4096 Jun 21 12:01 .
drwxr-xr-x 1 root root 4096 Jun 21 12:01 ..
drwxr-xr-x 2 root root 4096 Jun 21 12:01 configs
drwxr-xr-x 2 root root 4096 Jun 21 12:58 data
drwxr-xr-x 8 root root 4096 Jun 21 12:01 .git
-rw-r--r-- 1 root root   69 Jun 21 12:01 .gitignore
drwxr-xr-x 2 root root 4096 Jun 21 12:01 outputs
-rw-r--r-- 1 root root 3169 Jun 21 12:01 README.md
-rw-r--r-- 1 root root  303 Jun 21 12:01 requirements.txt
drwxr-xr-x 2 root root 4096 Jun 21 12:01 scripts
drwxr-xr-x 4 root root 4096 Jun 21 12:43 src


In [30]:
!ls -lh data

total 27M
-rw-r--r-- 1 root root 2.7M Mar 16  2020 TextVQA_0.5.1_test.json
-rw-r--r-- 1 root root  21M Mar 16  2020 TextVQA_0.5.1_train.json
-rw-r--r-- 1 root root 3.0M Mar 16  2020 TextVQA_0.5.1_val.json


In [31]:
!cat configs/default.yaml

# Default configuration. Override on the CLI where supported.

use_huggingface_dataset: true # Added this flag for Hugging Face integration

paths:
  data_root: data
  # TextVQA annotation files (download separately, see scripts/00_download.md)
  ann:
    train: 'data/TextVQA_0.5.1_train.json'
    val:   'data/TextVQA_0.5.1_val.json'
    test:  'data/TextVQA_0.5.1_test.json'
  # image folders (train_images, test_images from open-images subset)
  images:
    train: data/train_images
    val:   data/train_images   # val images live in the train_images folder for TextVQA 0.5.1
    test:  data/test_images
  ocr_cache: outputs/ocr_cache   # one json per split
  preds:     outputs/preds
  scores:    outputs/scores

ocr:
  engine: easyocr        # easyocr | tesseract
  langs: [en]
  max_tokens: 50         # cap OCR tokens fed to the reasoner
  min_confidence: 0.30

vlm:
  backend: blip2         # blip2 | api
  blip2_model: Salesforce/blip2-flan-t5-xl
  api_model: claude-sonnet-4-6     # only 

In [33]:
!python -c "from src.dataset import TextVQADataset; d=TextVQADataset('val'); print(len(d))"

5000


In [34]:
from src.dataset import TextVQADataset
d = TextVQADataset('val')
print(len(d))

5000


In [35]:
!sed -n '1,200p' src/ocr.py

"""OCR wrappers. Returns a list of {text, conf, bbox} tokens per image.

Two engines:
  - easyocr  : deep OCR, better on scene text, needs torch (GPU helps)
  - tesseract: fast CPU baseline, noisier on scene text -> good for the
               "noisy OCR" comparison the project asks for.
"""
from PIL import Image


class OCREngine:
    def read(self, image: Image.Image) -> list[dict]:
        raise NotImplementedError


class EasyOCR(OCREngine):
    def __init__(self, langs=("en",), gpu=False):
        import easyocr
        self.reader = easyocr.Reader(list(langs), gpu=gpu)

    def read(self, image: Image.Image) -> list[dict]:
        import numpy as np
        res = self.reader.readtext(np.array(image))
        out = []
        for bbox, text, conf in res:
            xs = [p[0] for p in bbox]
            ys = [p[1] for p in bbox]
            out.append({
                "text": text,
                "conf": float(conf),
                "bbox": [min(xs), min(ys), max(xs), max(ys)],


In [36]:
!pip install easyocr opencv-python

In [37]:
!ls data/train_images | head

ls: cannot access 'data/train_images': No such file or directory


In [39]:
!find data -maxdepth 2 -type d

data


In [40]:
!sed -n '1,200p' src/ocr.py

"""OCR wrappers. Returns a list of {text, conf, bbox} tokens per image.

Two engines:
  - easyocr  : deep OCR, better on scene text, needs torch (GPU helps)
  - tesseract: fast CPU baseline, noisier on scene text -> good for the
               "noisy OCR" comparison the project asks for.
"""
from PIL import Image


class OCREngine:
    def read(self, image: Image.Image) -> list[dict]:
        raise NotImplementedError


class EasyOCR(OCREngine):
    def __init__(self, langs=("en",), gpu=False):
        import easyocr
        self.reader = easyocr.Reader(list(langs), gpu=gpu)

    def read(self, image: Image.Image) -> list[dict]:
        import numpy as np
        res = self.reader.readtext(np.array(image))
        out = []
        for bbox, text, conf in res:
            xs = [p[0] for p in bbox]
            ys = [p[1] for p in bbox]
            out.append({
                "text": text,
                "conf": float(conf),
                "bbox": [min(xs), min(ys), max(xs), max(ys)],


In [41]:
!sed -n '1,250p' scripts/01_run_ocr.py

#!/usr/bin/env python
"""Run OCR over a split once and cache the result, so eval is fast and repeatable.

Usage:
    python scripts/01_run_ocr.py --split val --engine easyocr [--limit 50] [--gpu]
"""
import argparse
import json
import os
import sys

sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))

from tqdm import tqdm

from src.config import load_config
from src.dataset import TextVQADataset
from src.ocr import build_engine


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--split", default="val")
    ap.add_argument("--engine", default=None, help="easyocr|tesseract (default: config)")
    ap.add_argument("--limit", type=int, default=0)
    ap.add_argument("--gpu", action="store_true")
    args = ap.parse_args()

    cfg = load_config()
    engine_name = args.engine or cfg["ocr"]["engine"]
    ds = TextVQADataset(args.split, cfg, limit=args.limit)
    engine = build_engine(engine_name, tuple(cfg["ocr"]["langs"]), gpu=args.gpu)

    o

In [42]:
!pip install easyocr pytesseract opencv-python

In [43]:
!python scripts/01_run_ocr.py \
    --split val \
    --engine easyocr \
    --limit 100

Using CPU. Note: This module is much faster with a GPU.
Progress: |██████████████████████████████████████████████████| 100.0% CompleteDownloading recognition model, please wait. This may take several minutes depending upon your network connection.
OCR[easyocr]: 100% 100/100 [00:00<00:00, 41052.21it/s]
wrote outputs/ocr_cache/val_easyocr.json  (64 images)


# **Inspect the OCR output**

In [45]:
import json

with open("outputs/ocr_cache/val_easyocr.json") as f:
    data = json.load(f)

first_key = list(data.keys())[0]
print(first_key)
print(data[first_key])

003a8ae2ef43b901
[]


In [46]:
import json

with open("outputs/ocr_cache/val_easyocr.json") as f:
    data = json.load(f)

non_empty = 0

for img_id, tokens in data.items():
    if len(tokens) > 0:
        non_empty += 1
        print("Example image:", img_id)
        print(tokens[:5])   # first few OCR tokens
        break

print("Total images:", len(data))
print("Images with OCR text:", sum(len(v) > 0 for v in data.values()))

Total images: 64
Images with OCR text: 0


In [47]:
!find data -type d | sort

data
